# Surgery Phase Detection — Data Exploration

This notebook explores the Cholec80 dataset structure, phase distributions,
and temporal patterns in surgical phase annotations.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

from surgery_phase_detection import PHASE_NAMES, NUM_CLASSES
from surgery_phase_detection.utils.visualization import PHASE_COLORS

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

## 1. Load Phase Annotations

Load and inspect the phase annotation files from the Cholec80 dataset.

In [ ]:
ANNOTATIONS_DIR = Path('../data/phase_annotations/')

# Load all annotation files
all_annotations = {}

for vid_id in range(1, 81):
    video_name = f'video{vid_id:02d}'
    ann_file = ANNOTATIONS_DIR / f'{video_name}-phase.txt'
    
    if ann_file.exists():
        df = pd.read_csv(ann_file, sep='\t', header=0, names=['frame', 'phase'])
        all_annotations[video_name] = df

print(f'Loaded annotations for {len(all_annotations)} videos')

if all_annotations:
    example = list(all_annotations.values())[0]
    print(f'\nExample annotation (first video):')
    print(example.head(10))
    print(f'\nShape: {example.shape}')
else:
    print('No annotation files found. Please download the Cholec80 dataset first.')
    print(f'Expected location: {ANNOTATIONS_DIR}')

## 2. Phase Distribution Analysis

Analyze how surgical phases are distributed across the dataset.

In [ ]:
if all_annotations:
    # Aggregate all phase labels
    all_phases = np.concatenate([df['phase'].values for df in all_annotations.values()])
    
    # Count per phase
    phase_counts = Counter(all_phases)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar chart
    phases = sorted(phase_counts.keys())
    counts = [phase_counts[p] for p in phases]
    names = [PHASE_NAMES[p] for p in phases]
    colors = [PHASE_COLORS[p] for p in phases]
    
    ax1.bar(names, counts, color=colors, edgecolor='black', linewidth=0.5)
    ax1.set_ylabel('Frame Count')
    ax1.set_title('Phase Distribution (All Videos)')
    ax1.tick_params(axis='x', rotation=45)
    
    # Pie chart
    ax2.pie(counts, labels=names, colors=colors, autopct='%1.1f%%', startangle=90)
    ax2.set_title('Phase Proportions')
    
    plt.tight_layout()
    plt.show()
    
    print('Phase counts:')
    for p in phases:
        print(f'  {PHASE_NAMES[p]}: {phase_counts[p]:>8,} frames ({phase_counts[p]/len(all_phases)*100:.1f}%)')
else:
    print('No annotations loaded.')

## 3. Video Duration Analysis

Analyze the duration and phase structure of individual videos.

In [ ]:
if all_annotations:
    video_stats = []
    
    for name, df in all_annotations.items():
        duration_frames = len(df)
        duration_seconds = duration_frames / 25  # Assuming 25 fps
        n_transitions = np.sum(df['phase'].values[1:] != df['phase'].values[:-1])
        unique_phases = df['phase'].nunique()
        
        video_stats.append({
            'video': name,
            'frames': duration_frames,
            'duration_min': duration_seconds / 60,
            'transitions': n_transitions,
            'unique_phases': unique_phases,
        })
    
    stats_df = pd.DataFrame(video_stats)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].hist(stats_df['duration_min'], bins=20, color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Duration (minutes)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Video Duration Distribution')
    
    axes[1].hist(stats_df['transitions'], bins=20, color='coral', edgecolor='black')
    axes[1].set_xlabel('Number of Phase Transitions')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Phase Transitions per Video')
    
    axes[2].hist(stats_df['unique_phases'], bins=range(1, 9), color='mediumseagreen', edgecolor='black', align='left')
    axes[2].set_xlabel('Unique Phases')
    axes[2].set_ylabel('Count')
    axes[2].set_title('Unique Phases per Video')
    
    plt.tight_layout()
    plt.show()
    
    print(stats_df.describe())
else:
    print('No annotations loaded.')

## 4. Phase Timeline Visualization

Visualize the temporal structure of phases across sample videos.

In [ ]:
if all_annotations:
    # Show first 5 videos
    n_show = min(5, len(all_annotations))
    fig, axes = plt.subplots(n_show, 1, figsize=(16, 2 * n_show), sharex=False)
    
    for idx, (name, df) in enumerate(list(all_annotations.items())[:n_show]):
        ax = axes[idx] if n_show > 1 else axes
        phases = df['phase'].values
        time_min = np.arange(len(phases)) / 25 / 60  # frames -> minutes
        
        for c in range(NUM_CLASSES):
            mask = phases == c
            if np.any(mask):
                ax.fill_between(time_min, 0, 1, where=mask,
                              color=PHASE_COLORS[c], alpha=0.8)
        
        ax.set_ylabel(name, fontsize=9)
        ax.set_yticks([])
        ax.set_xlim(0, time_min[-1])
    
    axes[-1].set_xlabel('Time (minutes)')
    
    # Legend
    handles = [plt.Rectangle((0,0),1,1, fc=PHASE_COLORS[i]) for i in range(NUM_CLASSES)]
    fig.legend(handles, PHASE_NAMES, loc='center right', bbox_to_anchor=(1.2, 0.5), fontsize=8)
    
    fig.suptitle('Phase Timelines (Sample Videos)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No annotations loaded.')

## 5. Phase Transition Matrix

Analyze which phase transitions are most common.

In [ ]:
if all_annotations:
    transition_matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    
    for df in all_annotations.values():
        phases = df['phase'].values
        for i in range(len(phases) - 1):
            if phases[i] != phases[i+1]:
                transition_matrix[phases[i], phases[i+1]] += 1
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        transition_matrix,
        annot=True,
        fmt='d',
        cmap='YlOrRd',
        xticklabels=PHASE_NAMES,
        yticklabels=PHASE_NAMES,
        ax=ax,
        linewidths=0.5,
    )
    ax.set_xlabel('To Phase')
    ax.set_ylabel('From Phase')
    ax.set_title('Phase Transition Matrix (across all videos)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No annotations loaded.')

## 6. Class Imbalance Analysis

Compute class weights that will be used during training to handle imbalance.

In [ ]:
if all_annotations:
    all_phases = np.concatenate([df['phase'].values for df in all_annotations.values()])
    phase_counts = Counter(all_phases)
    total = len(all_phases)
    num_classes = max(phase_counts.keys()) + 1
    
    # Inverse frequency weights
    weights = {}
    for c in range(num_classes):
        count = phase_counts.get(c, 1)
        weights[PHASE_NAMES[c]] = total / (num_classes * count)
    
    print('Recommended class weights (inverse frequency):')
    for name, w in weights.items():
        print(f'  {name}: {w:.4f}')
    
    # Visualize imbalance ratio
    max_count = max(phase_counts.values())
    ratios = {PHASE_NAMES[c]: max_count / phase_counts[c] for c in sorted(phase_counts.keys())}
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(ratios.keys(), ratios.values(), color=PHASE_COLORS[:num_classes], edgecolor='black')
    ax.set_ylabel('Imbalance Ratio (vs majority class)')
    ax.set_title('Class Imbalance Ratios')
    ax.axhline(y=1, color='red', linestyle='--', alpha=0.5)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No annotations loaded.')